In [ ]:
import gradio as gr
import torch
from transformers import pipeline, AutoProcessor, AutoModelForSpeechSeq2Seq
import librosa
import soundfile as sf
import numpy as np
import os
import sys

# --- 1. CORE FUNCTIONALITY ---

# Docstring for the entire script
"""
Speech Recognition System (Speech-to-Text) with Gradio UI.

This application provides a user-friendly interface to transcribe audio
from uploaded files or live microphone input using a pre-trained
Hugging Face Transformers model (Whisper 'tiny').

Features:
- Transcribes short audio clips to text.
- Supports common audio formats (.mp3, .wav).
- Robust error handling for audio processing.
- Simple web interface built with Gradio.
- Modular code structure for readability and maintainability.
"""

def load_speech_model(model_name: str = "openai/whisper-tiny") -> pipeline:
    """
    Loads a pre-trained Speech-to-Text model from the Hugging Face Transformers library.

    Args:
        model_name (str): The name of the model to load (e.g., "openai/whisper-tiny").

    Returns:
        transformers.pipelines.pipeline: A Hugging Face pipeline for Automatic Speech Recognition.

    Raises:
        OSError: If the model cannot be loaded (e.g., network issues, invalid model name).
        Exception: For other unexpected errors during model loading.
    """
    try:
        # Determine the device for model inference (GPU if available, else CPU)
        device = "cuda:0" if torch.cuda.is_available() else "cpu"
        print(f"Loading model '{model_name}' on device: {device}")

        # Initialize the ASR pipeline, which simplifies model and processor handling.
        # chunk_length_s and stride_length_s are crucial for handling longer audios efficiently.
        asr_pipeline = pipeline(
            "automatic-speech-recognition",
            model=model_name,
            chunk_length_s=30, # Process audio in 30-second chunks
            stride_length_s=5, # Overlap chunks by 5 seconds to ensure continuity
            device=device,
            # Use float16 on GPU for faster inference, float32 on CPU
            torch_dtype=torch.float16 if device == "cuda:0" else torch.float32
        )
        print(f"Model '{model_name}' loaded successfully.")
        return asr_pipeline
    except OSError as e:
        # Specific error for model loading issues (e.g., file not found, network error)
        print(f"Error loading model {model_name}: {e}", file=sys.stderr)
        raise OSError(f"Could not load model '{model_name}'. Check network connection or model name. {e}")
    except Exception as e:
        # Catch any other unexpected errors during model initialization
        print(f"An unexpected error occurred while loading the model: {e}", file=sys.stderr)
        raise

# Global variable to store the ASR pipeline to avoid reloading it on every transcription request
asr_pipeline = None

def transcribe_audio(audio_input: str | tuple) -> str:
    """
    Transcribes an audio input, which can be a file path or a live recording from Gradio.

    Args:
        audio_input (str | tuple): Path to the audio file (str) or a tuple
                                   (sample_rate, numpy_array) from Gradio's microphone input.

    Returns:
        str: The transcribed text, or an error message if transcription fails.

    Raises:
        ValueError: If audio_input is invalid or of an unsupported type.
        FileNotFoundError: If the provided audio file path does not exist.
        Exception: For other unexpected transcription errors.
    """
    global asr_pipeline
    # Load the model if it hasn't been loaded yet (first call)
    if asr_pipeline is None:
        try:
            asr_pipeline = load_speech_model()
        except Exception as e:
            return f"Error: Could not initialize speech model. {e}"

    # Handle cases where no audio is provided
    if audio_input is None:
        return "Error: No audio input provided. Please upload a file or record audio."

    try:
        # Gradio's Audio component can return a tuple (sample_rate, numpy_array) for microphone
        # input or a file path (str) for uploaded files.
        if isinstance(audio_input, tuple):
            sr, y = audio_input
            if y is None or len(y) == 0:
                return "Error: Empty audio recording detected." # Handle empty mic input

            # Convert audio data to float32 and normalize, as expected by Hugging Face models.
            # This ensures consistent input format regardless of source.
            audio_array = y.astype(np.float32)
            if np.max(np.abs(y)) > 0: # Avoid division by zero for silent audio
                audio_array /= np.max(np.abs(y))

            # The pipeline expects a dictionary for raw audio data
            audio_for_pipeline = {"sampling_rate": sr, "raw": audio_array}

        elif isinstance(audio_input, str):
            # Check if the audio file exists for file path inputs
            if not os.path.exists(audio_input):
                raise FileNotFoundError(f"Audio file not found: {audio_input}")

            # The pipeline can directly take a file path for uploaded files
            audio_for_pipeline = audio_input
        else:
            # Handle unexpected input types
            raise ValueError("Unsupported audio input type. Expected file path or (sample_rate, numpy_array).")

        print("Starting transcription...")
        # Perform the actual transcription using the loaded ASR pipeline.
        # The pipeline handles pre-processing (resampling, tokenization) and model inference.
        prediction = asr_pipeline(audio_for_pipeline)
        transcribed_text = prediction["text"]
        print(f"Transcription complete: {transcribed_text}")
        return transcribed_text.strip()

    except FileNotFoundError as e:
        return f"Error: Audio file not found. {e}"
    except Exception as e:
        # Generic error handling for transcription failures (e.g., corrupted file, model issues).
        print(f"Transcription error: {e}", file=sys.stderr)
        return f"Error during transcription: {e}. Please ensure the audio input is valid and supported (.mp3, .wav)."

# --- 2. USER INTERFACE (UI) ---

def launch_app():
    """
    Launches the Gradio web application for the Speech Recognition System.
    The UI allows users to upload audio files or record live speech and displays the transcription.
    """
    print("Initializing Gradio UI...")
    # Define the Gradio Interface components.
    # gr.Audio is versatile, supporting both file uploads and microphone input.
    # type="filepath" returns the path to a temporary file for uploads.
    # type="numpy" returns (sample_rate, numpy_array) for microphone recordings.
    iface = gr.Interface(
        fn=transcribe_audio, # The function to be called when input changes
        inputs=gr.Audio(
            sources=["upload", "microphone"], # Allow both file upload and microphone input
            type="filepath", # For uploads, provide file path; for microphone, pipeline handles numpy conversion
            label="Upload Audio or Record from Microphone (.mp3, .wav)"
        ),
        outputs=gr.Textbox(label="Transcribed Text", lines=5, placeholder="Transcription will appear here..."),
        title="🎙️ Speech Recognition System (Whisper Tiny)",
        description="""
        This application transcribes speech from audio files or live microphone input using
        the OpenAI Whisper 'tiny' model via Hugging Face Transformers.
        Upload an .mp3 or .wav file, or record directly from your microphone.
        """,
        allow_flagging="never", # Disable the flagging feature common in Gradio demos
    )
    print("Gradio UI launched. Running locally...")
    # Launch the Gradio app. debug=True provides useful logs, share=True generates a public link.
    iface.launch(debug=True, share=True)

# --- 4. PORTFOLIO ASSETS ---
# The content for 'requirements.txt' and 'README.md' is provided as multi-line strings
# within this Python script. Users can copy these strings and save them into their
# respective files for a complete project setup.

REQUIREMENTS_TXT_CONTENT = """
torch>=1.10.0
transformers>=4.20.0
gradio>=3.0
librosa>=0.9.0 # Used indirectly by transformers for some audio operations
soundfile>=0.12.0 # Used for reading/writing audio files, also often a dependency
numpy>=1.20.0
"""

README_MD_CONTENT = """


"""

# --- Main execution block ---
if __name__ == "__main__":
    print("CODTECH IT SOLUTIONS - Speech Recognition System")
    print("Initiating application startup...")

    # Optional: Print requirements and README content for the user to save
    print("\n--- BEGIN requirements.txt CONTENT ---")
    print(REQUIREMENTS_TXT_CONTENT.strip())
    print("--- END requirements.txt CONTENT ---\n")
    print("Please save the above content into a file named 'requirements.txt'.")

    print("\n--- BEGIN README.md CONTENT ---")
    print(README_MD_CONTENT.strip())
    print("--- END README.md CONTENT ---\n")
    print("Please save the above content into a file named 'README.md'.")

    # Launch the Gradio application
    launch_app()